# План проекта
В проекте вы реализуете pretrain и posttrain этапы обучения LLM. Выполняйте проект в Jupyter Notebook на ВМ. Выполнение заданий проекта займёт от 5 до 8 часов, не считая времени на обучение. 


## Pretrain
Претрейн — самый ресурсоёмкий этап обучения LLM. Чтобы полноценно обучить даже небольшую модель (менее 1B), понадобится более 10к GPU-часов на A100. Чтобы не тратить недели на обучение, но отработать ключевые приёмы, в проекте вы выполните упрощённую задачу. 

При полноценном претрейне модель учится обобщать знания из данных, на которых происходило обучение, чтобы потом извлекать эти знания по текстовым запросам уже после обучения. Упростим задачу — научим модель только структуре языка. 
Сосредоточимся на одном узком домене — текстах произведений русской литературы — и обучим модель продолжать фразы из этого домена разумным текстом. 

### Шаги этапа
1. Скачайте данные из [репозитория](https://github.com/JoannaBy/RussianNovels/tree/master/corpus) и упакуйте их в один датасет. Вам понадобятся все произведения из репозитория.

In [3]:
import os
from github_downloader import download_from_github
url = "https://github.com/JoannaBy/RussianNovels/tree/master/corpus"
current_dir = os.getcwd()
dataset_folder = os.path.join(current_dir, "dataset")
try:
    os.makedirs(dataset_folder, exist_ok=True)
    download_from_github(
        url=url,
        dest_folder=dataset_folder,
    )
    print(f"Датасет успешно загружен в: {dataset_folder}")    
except Exception as e:
    print(f"Произошла ошибка при загрузке: {e}")

Downloaded: /home/ubuntu/project/dataset/Bulgakov_BelayaGvardiya.txt
Downloaded: /home/ubuntu/project/dataset/Bulgakov_Diavoliada.txt
Downloaded: /home/ubuntu/project/dataset/Bulgakov_Master.txt
Downloaded: /home/ubuntu/project/dataset/Bulgakov_RokovyeYaytsa.txt
Downloaded: /home/ubuntu/project/dataset/Bulgakov_TeatralnyjRoman.txt
Downloaded: /home/ubuntu/project/dataset/Bulgakov_ZapiskiYonogoVracha.txt
Downloaded: /home/ubuntu/project/dataset/Chekhov_Dama.txt
Downloaded: /home/ubuntu/project/dataset/Chekhov_Dyadya.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevski_Biesy.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevski_Idiot.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevsky_BednyeLyudi.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevsky_Karamazow1.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevsky_Karamazow2.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevsky_Karamazow3.txt
Downloaded: /home/ubuntu/project/dataset/Dostoyevsky_Karamazow4.txt
Download

2. Проведите препроцессинг данных:
   - Очистите их от дубликатов.
   - Очистите от предложений с буквами не из кириллицы.
   - Обработайте повторяющуюся пунктуацию и т. д.
   - Разбейте на чанки поменьше, чтобы можно было добавить `<bos>` и `<eos>` токены в соответствии с обучаемой длиной контекста.

#### Препроцессинг

In [4]:
import os
import re
import glob
import json
from typing import List, Tuple, Set
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm


def load_text_files(data_dir: str) -> List[str]:
    """
    Загружает все текстовые файлы из указанной директории.
    Возвращает список строк (каждая строка - содержимое файла).
    """
    texts = []
    file_pattern = os.path.join(data_dir, "*.txt")
    for file_path in tqdm(glob.glob(file_pattern)):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                text = f.read()
                texts.append(text)
        except Exception as e:
            print(f"Ошибка при чтении файла {file_path}: {e}")
    return texts


def split_into_sentences(text: str) -> List[str]:
    """
    Разбивает текст на предложения по точкам, восклицательным и вопросительным знакам.
    Учитывает многоточие и сокращения (например, "т.д.").
    Упрощённый подход.
    """
    # Заменяем многоточие на специальный маркер, чтобы не разбивать по нему
    text = re.sub(r'\.\.\.', '…', text)
    # Заменяем сокращения с точками
    abbreviations = [r'т\.д\.', r'т\.п\.', r'др\.', r'пр\.', r'г\.', r'см\.', r'ст\.', r'кн\.', r'ч\.', r'с\.']
    for abbr in abbreviations:
        text = re.sub(abbr, abbr.replace('.', '@'), text)
    
    # Разбиваем по . ! ? … (многоточие)
    sentences = re.split(r'(?<=[.!?…]) +', text)
    
    # Восстанавливаем сокращения
    restored = []
    for sent in tqdm(sentences):
        sent = sent.replace('@', '.')
        restored.append(sent.strip())
    
    # Убираем пустые предложения
    restored = [s for s in restored if s]
    return restored


def filter_cyrillic_sentences(sentences: List[str]) -> List[str]:
    """
    Оставляет только предложения, состоящие преимущественно из кириллических символов,
    пробелов, знаков пунктуации и цифр.
    """
    # Регулярное выражение для кириллических символов, пробелов, пунктуации и цифр
    cyrillic_pattern = re.compile(r'^[а-яёА-ЯЁ0-9\s\.,!?;:"\'\-–—()…]+$')
    filtered = []
    for sent in sentences:
        if cyrillic_pattern.match(sent):
            filtered.append(sent)
        else:
            # Можно также проверить процент кириллических символов
            # но для простоты используем строгое соответствие
            pass
    return filtered


def deduplicate_sentences(sentences: List[str]) -> List[str]:
    """
    Удаляет дубликаты предложений (точное совпадение).
    """
    seen = set()
    unique = []
    for sent in sentences:
        if sent not in seen:
            seen.add(sent)
            unique.append(sent)
    return unique


def clean_punctuation(text: str) -> str:
    """
    Очищает повторяющуюся пунктуацию (например, "!!!", "??", ",,").
    Заменяет множественные пробелы на один, удаляет неразрывные пробелы.
    """
    # Убираем повторяющиеся знаки препинания (оставляем один)
    text = re.sub(r'([!?])\1+', r'\1', text)  # !! -> !
    text = re.sub(r'(,)\1+', r'\1', text)     # ,, -> ,
    text = re.sub(r'(\.)\1+', r'\1', text)    # .. -> . (хотя .. обычно не встречается)
    # Убираем повторяющиеся дефисы, тире
    text = re.sub(r'(-)\1+', r'\1', text)
    text = re.sub(r'(—)\1+', r'\1', text)
    # Заменяем все whitespace символы (включая неразрывные пробелы, табуляции) на обычный пробел
    text = re.sub(r'\s+', ' ', text)
    # Удаляем пробелы перед знаками препинания (кроме открывающих скобок)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    # Удаляем пробелы после открывающих скобок и перед закрывающими
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    return text.strip()


def chunk_text(sentences: List[str], max_chunk_size: int = 512) -> List[str]:
    """
    Объединяет предложения в чанки примерно max_chunk_size символов.
    Добавляет специальные токены <bos> и <eos> в начале и конце каждого чанка.
    """
    chunks = []
    current_chunk = []
    current_length = 0
    
    for sent in tqdm(sentences, "Формируем чанки: "):
        sent_len = len(sent)
        if current_length + sent_len + 1 <= max_chunk_size:  # +1 для пробела
            current_chunk.append(sent)
            current_length += sent_len + 1
        else:
            if current_chunk:
                chunk_text = ' '.join(current_chunk)
                chunk_text = f"<bos> {chunk_text} <eos>"
                chunks.append(chunk_text)
            # Начинаем новый чанк с текущим предложением
            current_chunk = [sent]
            current_length = sent_len
    
    # Добавляем последний чанк
    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        chunk_text = f"<bos> {chunk_text} <eos>"
        chunks.append(chunk_text)
    
    return chunks

/home/ubuntu/project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


загрузим данные посмотрим на текст из списка текстов

In [5]:
import os
current_dir = os.getcwd()
dataset_folder = os.path.join(current_dir, "dataset")
text_files = load_text_files(dataset_folder)
print(text_files[0][:1000])

 44%|████▎     | 47/108 [00:00<00:00, 449.46it/s]

100%|██████████| 108/108 [00:00<00:00, 477.64it/s]

КАПИТАНСКАЯ ДОЧКА

Береги честь смолоду.

Пословица.

ГЛАВА I

СЕРЖАНТ ГВАРДИИ

-- Был бы гвардии он завтра ж капитан.

-- Того не надобно; пусть в армии послужит.

-- Изрядно сказано! пускай его потужит...

. . . . . . . . . . . . . . .

Да кто его отец?

Княжнин.

   Отец мой Андрей Петрович Гринев в молодости своей служил при графе Минихе и вышел в отставку премьер-майором в 17.. году. С тех пор жил он в своей Симбирской деревне, где и женился на девице Авдотье Васильевне Ю., дочери бедного тамошнего дворянина. Нас было девять человек детей. Все мои братья и сестры умерли во младенчестве.
   Матушка была еще мною брюхата, как уже я был записан в Семеновский полк сержантом, по милости майора гвардии князя Б., близкого нашего родственника. Если бы паче всякого чаяния матушка родила дочь, то батюшка объявил бы куда следовало о смерти неявившегося сержанта, и дело тем бы и кончилось. Я считался в отпуску до окончания наук. В то время воспитывались мы не по-нонешнему. С пятилетнего возра

сконкатенируем текста и разобъем их на предложения

In [6]:
sentences = split_into_sentences("\n".join(text_files))
print(f"Предложений: {len(sentences)}")
print()
for i in range(10):
    print(f"Предложение {i}")
    print(sentences[i])

100%|██████████| 379440/379440 [00:00<00:00, 2417339.23it/s]


Предложений: 379440

Предложение 0
КАПИТАНСКАЯ ДОЧКА

Береги честь смолоду.

Пословица.

ГЛАВА I

СЕРЖАНТ ГВАРДИИ

-- Был бы гвардии он завтра ж капитан.

-- Того не надобно; пусть в армии послужит.

-- Изрядно сказано!
Предложение 1
пускай его потужит…

.
Предложение 2
.
Предложение 3
.
Предложение 4
.
Предложение 5
.
Предложение 6
.
Предложение 7
.
Предложение 8
.
Предложение 9
.


In [7]:
processed = filter_cyrillic_sentences(sentences)
print(f"Предложений после фильтрации кириллицы: {len(processed)}")

processed = [clean_punctuation(sentence) for sentence in processed]
processed = deduplicate_sentences(processed)
print(f"Предложений после дедупликации: {len(processed)}")

chunks = chunk_text(processed, max_chunk_size=512)
print(f"Чанков: {len(chunks)}")   

Предложений после фильтрации кириллицы: 340806
Предложений после дедупликации: 326809


Формируем чанки: 100%|██████████| 326809/326809 [00:00<00:00, 1719587.24it/s]

Чанков: 82048


3. Создайте и обучите собственный токенизатор на полученных данных. Размер словаря выберите небольшим: при обучении только на рассмотренных текстах — около 3к токенов. В рассматриваемых данных язык намного менее разнообразен, чем в совокупных данных, поэтому крупные токенизаторы от реальных LLM могут не подойти. 

   При создании токенизатора можете ориентироваться на [материал huggingface.co](https://huggingface.co/learn/llm-course/ru/chapter6/8). Рекомендуем использовать BPE.

In [70]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def train_bpe_tokenizer(texts: List[str], vocab_size: int = 3000, save_path: str = "tokenizer.json"):
    """
    Обучает BPE токенизатор на предоставленных текстах.
    Сохраняет токенизатор в файл.
    """    
    # Инициализируем токенизатор с BPE моделью
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
    tokenizer.decoder = ByteLevelDecoder()
    
    # Создаём тренера с указанным размером словаря
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>", "<mask>"],
        min_frequency=2
    )
    
    # Обучаем на текстах (передаём список строк)
    tokenizer.train_from_iterator(texts, trainer)
    
    # Сохраняем токенизатор
    tokenizer.save(save_path)
    print(f"Токенизатор сохранён в {save_path}")

    # Возвращаем токенизатор для дальнейшего использования
    return tokenizer

In [71]:
from transformers import PreTrainedTokenizerFast


tokenizer = train_bpe_tokenizer(sentences, vocab_size=3000, save_path="tokenizer.json")
tokenizer_fast = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<unk>",
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    mask_token="<mask>",
)

print(f"Размер словаря: {len(tokenizer.get_vocab())}")




Токенизатор сохранён в tokenizer.json
Размер словаря: 3000


In [72]:
def infer_tokenizer(tokenizer, texts):
    """Тестирует токенизатор на наборе предложений."""
    for i, text in enumerate(texts):
        encoded_ids = tokenizer.encode(text).ids
        print(f"\nПредложение {i+1}: {text[:80]}...")
        print(f"   Токены: {[tokenizer.decode([id]) for id in encoded_ids[:20]]}{'...' if len(encoded_ids) > 20 else ''}")
        print(f"   IDs:    {encoded_ids[:10]}{'...' if len(encoded_ids) > 10 else ''}")
        print(f"   Количество токенов: {len(encoded_ids)}")


def specail_tokens_check(tokenizer): 
    """Проверка специальных токенов."""
    special_tokens = ["<bos>", "<eos>", "<pad>", "<unk>", "<mask>"]
    for tok in special_tokens:
        try:
            id_ = tokenizer.token_to_id(tok)
            print(f"  {tok}: ID = {id_}")
        except:
            print(f"  {tok}: не найден в словаре")


def count_unk(tokenizer, texts):
    """Тестирует встречаемость токена <unk>"""
    token_cnt, unk_cnt = 0, 0
    for text in tqdm(texts, "Скан текстов"):
        tokens = tokenizer.encode(text).tokens
        token_cnt += len(tokens)
        unk_cnt += len([t for t in tokens if t == '<unk>'])
    print(f"Кол-во <unk> токенов в текстах: {unk_cnt}")
    print(f"Доля <unk> токенов в текстах:   {100 * unk_cnt/token_cnt:.4f}")


print("\nПример работы токенизатора")
infer_tokenizer(tokenizer, chunks[10000:10003])

print("\nПроверка специальных токенов")
specail_tokens_check(tokenizer)

print("\nПроверка unk токенов")
count_unk(tokenizer, chunks)


Пример работы токенизатора

Предложение 1: <bos> Что? - Пришла ко мне и объясняет, что они с Алексеем любят друг друга… Как...
   Токены: ['', ' Что', '?', ' -', ' При', 'шла', ' ко', ' мне', ' и', ' объяс', 'ня', 'ет', ',', ' что', ' они', ' с', ' Алексе', 'ем', ' люб', 'ят']...
   IDs:    [2, 772, 35, 219, 1569, 1291, 373, 485, 200, 1786]...
   Количество токенов: 148

Предложение 2: <bos> Лиля, ну кому еще, рыжая, рыжая! Литта онемела и почувствовала себя оглупе...
   Токены: ['', ' Л', 'ил', 'я', ',', ' ну', ' к', 'ому', ' еще', ',', ' ры', 'ж', 'ая', ',', ' ры', 'ж', 'ая', '!', ' Лит', 'та']...
   IDs:    [2, 446, 223, 187, 16, 1833, 204, 422, 501, 16]...
   Количество токенов: 172

Предложение 3: <bos> Сижу, не знаю, сколько времени прошло… Потом под окном голоса, выглянула,-...
   Токены: ['', ' С', 'ижу', ',', ' не', ' знаю', ',', ' сколько', ' времени', ' прош', 'ло', '…', ' Потом', ' под', ' ок', 'ном', ' голос', 'а', ',', ' выг']...
   IDs:    [2, 324, 1975, 16, 235, 1063, 

Скан текстов: 100%|██████████| 82048/82048 [00:24<00:00, 3385.41it/s]

Кол-во <unk> токенов в текстах: 0
Доля <unk> токенов в текстах:   0.0000


In [73]:
text = "Тестовое сообщение"
encoded = tokenizer.encode(text)
print(tokenizer.decode(encoded.ids, skip_special_tokens=True,))

 Тестовое сообщение


4. Токенизируйте данные и подготовьте их к претрейну с длиной контекста 512 токенов в виде экземпляра класса `transformers.Dataset`.

In [74]:
from datasets import Dataset


def tokenize(tokenizer):
    def _tokenize(samples):
        return tokenizer(
            samples["chunks"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
    return _tokenize


dataset = Dataset.from_dict({"chunks": chunks}).map(
    tokenize(tokenizer_fast),
    batched=True,
    remove_columns=["chunks"]
)
dataset

Map: 100%|██████████| 82048/82048 [00:21<00:00, 3802.42 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 82048
})

5. Инициализируйте модель ~150M параметров c произвольной decoder-only архитектурой трансформера. 

   Например, можно рассмотреть LlamaConfig с параметрами `hidden_size=1024, intermediate_size=1536, num_hidden_layers=16, num_attention_heads=16, num_key_value_heads=8`.

In [106]:
import torch
from dataclasses import dataclass
from transformers import LlamaConfig, LlamaForCausalLM
from torchinfo import summary


def init_model(tokenizer, model_config, device=None) -> LlamaForCausalLM:
    llama_config = LlamaConfig(
        vocab_size          = len(tokenizer.get_vocab()),
        hidden_size         = model_config.hidden_size,     
        intermediate_size   = model_config.intermediate_size,
        num_hidden_layers   = model_config.num_hidden_layers,
        num_attention_heads = model_config.num_attention_heads,
        num_key_value_heads = model_config.num_key_value_heads,
        attention_dropout   = model_config.attention_dropout,
        hidden_dropout      = model_config.hidden_dropout,
        pad_token_id        = tokenizer.pad_token_id,
        bos_token_id        = tokenizer.bos_token_id,
        eos_token_id        = tokenizer.eos_token_id,
        use_cache           = True,
    )
    model = LlamaForCausalLM(llama_config)
    
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    print(f"Модель инициализирована на устройстве: {device}")
    if device == "cuda":
        print(f"Устройство: {torch.cuda.get_device_name(0)}")

    return model


@dataclass
class ModelConfig:
    hidden_size: int = 1024
    intermediate_size: int = 1536
    num_hidden_layers: int = 16
    num_attention_heads: int = 16
    num_key_value_heads: int = 8
    attention_dropout: float = 0.1
    hidden_dropout: float = 0.1

model = init_model(tokenizer_fast, ModelConfig)

input_data = torch.randint(
    0,
    tokenizer_fast.vocab_size,
    size=(1, 512),
    device=next(model.parameters()).device,
    dtype=torch.long
)

summary(
    model,
    input_data=input_data,
    col_names=[
        "input_size",
        "output_size",
        "num_params",
    ],
    depth=3,
    device=next(model.parameters()).device,
)

Модель инициализирована на устройстве: cuda
Устройство: NVIDIA A100-SXM4-80GB


Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
LlamaForCausalLM                              [1, 512]                  --                        --
├─LlamaModel: 1-1                             --                        --                        --
│    └─Embedding: 2-1                         [1, 512]                  [1, 512, 1024]            3,072,000
│    └─LlamaRotaryEmbedding: 2-2              [1, 512, 1024]            [1, 512, 64]              --
│    └─ModuleList: 2-3                        --                        --                        --
│    │    └─LlamaDecoderLayer: 3-1            [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─LlamaDecoderLayer: 3-2            [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─LlamaDecoderLayer: 3-3            [1, 512, 1024]            [1, 512, 1024]            7,866,368
│    │    └─LlamaDecoderLayer: 3-4            [1, 512, 102

6. Чтобы оценить качество, используйте промпты:
```
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
] 
```

Подготовьте коллбэки для валидации качества на промптах. Реализуйте обучение с помощью `Trainer`. Обратите внимание на параметры регуляризации `weight_decay`. Используйте подходящий `batch_size` — в диапазоне 64—128.

In [ ]:
from dataclasses import dataclass
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, TrainerCallback, EarlyStoppingCallback
import math


TEST_PROMPTS = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]


class PromptEvalCallback(TrainerCallback):
    """
    Коллбэк для оценки качества генерации на тестовых промптах.
    Вызывается в конце каждой эпохи.
    """
    def __init__(self, tokenizer, model, prompts=TEST_PROMPTS, max_length=50, sample_params=None):
        self.tokenizer = tokenizer
        self.model = model
        self.prompts = prompts
        self.max_length = max_length
        self.sample_params = sample_params
        
    def on_epoch_end(self, args, state, control, **kwargs):
        """
        Генерирует тексты для каждого промпта и выводит их.
        """
        device = next(model.parameters()).device
        temperature = self.sample_params.temperature if self.sample_params else 0.8
        top_p = self.sample_params.top_p if self.sample_params else 0.9
        do_sample = self.sample_params.do_sample if self.sample_params else True

        self.model.eval()
        with torch.no_grad():
            for prompt in self.prompts:
                input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)
                output_ids = self.model.generate(
                    input_ids,
                    max_length=self.max_length,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                generated = self.tokenizer.decode(output_ids[0],
                                                  skip_special_tokens=True,
                                                  clean_up_tokenization_spaces=False)
                print(f"Промпт:         {prompt}")
                print(f"Сгенерировано:  {generated}\n")
        self.model.train()


class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            metrics["perplexity"] = math.exp(metrics["eval_loss"])
            print(f"Perplexity: {metrics['perplexity']:.2f}")


def run_sft(model, tokenizer, train_dataset, eval_dataset, sample_params):
    cfg = SFTConfig(
        output_dir="pretarined_sft",
        per_device_train_batch_size=16,
        logging_steps=1,
        max_length=512,
        report_to='none',
        run_name='SFT',
        num_train_epochs=10,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        weight_decay=5e-2,
        load_best_model_at_end=True,
        seed=42,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8,
    )
    prompt_callback = PromptEvalCallback(
        tokenizer,
        model,
        max_length=512,
        sample_params=sample_params,
    )
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.01
    )
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        # packing=True,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[
            PerplexityCallback(),
            prompt_callback,
            early_stopping_callback,
        ],
    )
    trainer.train()

In [ ]:
# dataset = dataset.select(range(12800))
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]


@dataclass
class SampleParams:
    temperature: float = 0.8
    top_p: float = 0.9
    do_sample: bool = True


run_sft(model, tokenizer_fast, train_dataset, eval_dataset, SampleParams)

Epoch,Training Loss,Validation Loss,
1,5.057900,4.971206,144.200716
2,4.396800,4.383810,80.142762
3,4.106300,4.098952,60.277102
4,3.704000,3.966344,52.791154
5,3.387800,3.917805,50.289923
6,3.146800,3.918119,50.305728
7,3.034700,3.944176,51.633797
8,2.827700,3.979134,53.470687


Промпт:         Все мысли, которые имеют огромные последствия
Сгенерировано:  Все мысли, которые имеют огромные последствия, и умилась вулы, на меня было на себя вувей, на коли, и в этом в немном махе, векарах, и вука мастерно, как бы нашем, но не может и сули, и на него, и что удается было не-то? 

Промпт:         Сила войска зависит от его духа
Сгенерировано:  Сила войска зависит от его духа палькускуюкоме товойания дорогскуюили домой словногоком мкомства дальшеах в мениямиыйскания, я уравая хода, вода, он опять не подушет ему входе и соображают пыльного, когда она за дверь в этом праздной черный табоче. - Валософ, что же? - сказал он, не может быть и, а за что-то, а он устроил, - и, что он подвигаясь, входили в этом. - Тетий, - я! - спросил он. - Не вы - вы! 

Промпт:         Мысль о том, что он принес страдания
Сгенерировано:  Мысль о том, что он принес страдания, не знаю, что за него, что ты мне не был ящику. 

Промпт:         Человек сознает себя свободным
Сгенерировано:  Человек

Ну вот такая модель обучилась с нуля. Пожалуй, надо больше данных и/или взять предобученный декодер

In [111]:
def test_prompts(prompts, tokenizer, max_length, sample_params=None):
    device = next(model.parameters()).device
    temperature = sample_params.temperature if sample_params else 0.8
    top_p = sample_params.top_p if sample_params else 0.9
    do_sample = sample_params.do_sample if sample_params else True

    model.eval()
    with torch.no_grad():
        for prompt in prompts:
            input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
            output_ids = model.generate(
                input_ids,
                max_length=max_length,
                temperature=temperature,
                top_p=top_p,
                do_sample=do_sample,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            generated = tokenizer.decode(output_ids[0],
                                                skip_special_tokens=True,
                                                clean_up_tokenization_spaces=False)
            print(f"Промпт:         {prompt}")
            print(f"Сгенерировано:  {generated}\n")

test_prompts(TEST_PROMPTS, tokenizer_fast, 512, SampleParams)

Промпт:         Все мысли, которые имеют огромные последствия
Сгенерировано:  Все мысли, которые имеют огромные последствия, хотя бы это было теперь и все то как бы их наше, что из них в них бы им подверженный отшедший от себя, и он не видел его видеть, что не дуэам - не понимал, что он не мог бы себе такими и не было, как он ни был в чеме, потому что не замечал, как они его не может и не решались, как он не поместился, потому что на себя, как с вином, и из них он не заметил, а только обстоятельствующего. 

Промпт:         Сила войска зависит от его духа
Сгенерировано:  Сила войска зависит от его духа ее счаоб, чтоб не приятно ему ничего. Она не решалась; она уже не было. - Как же? - спросил он. - Да, как же, может быть, это только одно дело? - спросил он. - И не знаю! - отвечала она и снова спросил: - Не знаю!. - Дело не любить! 

Промпт:         Мысль о том, что он принес страдания
Сгенерировано:  Мысль о том, что он принес страдания, сказалаят, когда она не возьмет меня, как она мен

### Идеи
Модель не супер справляется с генерацией. Да и на SFT она ушла в переобучение, поэтому за счет early stopping имеем лучшую, но не супер много проученную версию.

На практике думаю стоит:
- брать предобученную модель
- играть с архитектурой
- наливать больше данных

---
## Post-train SFT
Для SFT-этапа можно использовать значительно меньше данных, поэтому возьмём модель крупнее. Рассмотрим базовую модель Qwen2.5-0.5B, с которой вы встречались в уроках. Обучите её генерировать ответы на инструктивные русскоязычные вопросы.

Для оценки качества используйте такой набор вопросов:
```
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ] 
```

Чтобы повысить качество ответов, проведите SFT обучение на русскоязычном инструктивном датасете [d0rj/alpaca-cleaned-ru](https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru) в диалоговом формате. 

In [23]:
from datasets import load_dataset

dataset = load_dataset("d0rj/alpaca-cleaned-ru")
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 51760
    })
})

In [24]:
dataset = dataset["train"]

In [25]:
import torch
from transformers import AutoTokenizer

model_name = 'Qwen/Qwen2.5-0.5B'
tokenizer_model_name = f'{model_name}-Instruct'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

Опытный путем выяснил, что токенизатор Qwen вшивает в шаблон Qwen-специфичный системный промпт. Плюс тут ещё много логики для тулколинга. Решил переписать шаблон + упростить его.

In [26]:
tokenizer.chat_template = """
{%- if messages[0]['role'] == 'system' %}
{{- '<|im_start|>system\\n' + messages[0]['content'] + '<|im_end|>\\n' }}
{%- endif %}

{%- for message in messages %}
    {%- if not (loop.first and message['role'] == 'system') %}
        {{- '<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n' }}
    {%- endif %}
{%- endfor %}

{%- if add_generation_prompt %}
{{- '<|im_start|>assistant\\n' }}
{%- endif %}
"""

In [27]:
def row2messages(row):
    instruction = row["instruction"]
    if "input" in row and row["input"] != "":
        instruction += "\n" + row["input"]
    messages = [{"role": "user", "content": instruction}]
    if "output" in row:
        messages.append({"role": "assistant", "content": row["output"]})
    return messages       


dataset = dataset.select(range(3000))

dataset = dataset.map(lambda x: {'messages': row2messages(x)})
dataset = dataset.map(lambda x: {'text': tokenizer.apply_chat_template(x["messages"], tokenize=False)})

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

split

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output', 'messages', 'text'],
        num_rows: 2700
    })
    test: Dataset({
        features: ['input', 'instruction', 'output', 'messages', 'text'],
        num_rows: 300
    })
})

Для обучения 500М модели мне достался сервер с Tesla T4. GPU не из числа больших, поэтому я решил сначала обучить QLoRA, потому что полный SFT выглядит как задача на много часов.

Для PoC обучаюсь на небольшом подмножестве

In [14]:
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig


# qconf = BitsAndBytesConfig(load_in_8bit=True)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     dtype="auto",
#     quantization_config=qconf,
# )

from unsloth import FastLanguageModel
from torchinfo import summary

model, _ = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=256,
    load_in_4bit=True,
    load_in_8bit=False,
)

model = FastLanguageModel.get_peft_model(model,
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    # use_gradient_checkpointing=False, 
    use_gradient_checkpointing="unsloth",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)
device = next(model.parameters()).device
print(f"\nМодель инициализирована на устройстве: {device}")
if device.type == "cuda":
    print(f"Устройство: {torch.cuda.get_device_name(0)}\n")


# input_data = torch.randint(0, tokenizer.vocab_size, size=(1, 64), device=device, dtype=torch.long)

# summary(
#     model,
#     input_data=input_data,
#     col_names=[
#         "input_size",
#         "output_size",
#         "num_params",
#     ],
#     depth=4,
#     device=next(model.parameters()).device,
# )

model.print_trainable_parameters()

/home/ubuntu/project/.venv/lib/python3.10/site-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 05-20 20:32:15 [__init__.py:216] Automatically detected platform cuda.
ERROR 05-20 20:32:18 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.581 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-0.5b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.5.5 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.



Модель инициализирована на устройстве: cuda:0
Устройство: Tesla T4

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [17]:
from dataclasses import dataclass

@dataclass
class SampleParams:
    temperature: float = 0.8
    top_p: float = 0.9
    do_sample: bool = True


questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ]

Посмотрим что генерирует pretrained модель

In [ ]:
def test_prompts(model, tokenizer, prompts, max_length, sample_params=None):
    device = next(model.parameters()).device
    temperature = sample_params.temperature if sample_params else 0.8
    top_p = sample_params.top_p if sample_params else 0.9
    do_sample = sample_params.do_sample if sample_params else True

    model.eval()
    with torch.no_grad():
        for prompt in prompts:
            input_ids = tokenizer\
                .apply_chat_template(row2messages({"instruction": prompt}), return_tensors="pt")\
                .to(device)
            output_ids = model.generate(
                input_ids,
                max_new_tokens=256,
                temperature=temperature,
                top_p=top_p,
                do_sample=do_sample,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            generated_ids = output_ids[0]
            generated_tokens = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            print("---")
            print(f"Промпт:         {prompt}")
            print(f"Чат:\n{''.join(generated_tokens)}\n")


test_prompts(model, tokenizer, questions_rus, 512, SampleParams)

---
Промпт:         сколько планет в нашей солнечной системе?
Чат:
user
сколько планет в нашей солнечной системе?
каких планет нас солнечной системы насчитает?rów
сколько планет в нашей солнечной системе?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów
сколько планет нас солнечной системы насчитает?rów


---
Промпт:         расск

In [28]:
from dataclasses import dataclass
import torch.nn as nn
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, TrainerCallback, EarlyStoppingCallback
import math


class PromptEvalCallback(TrainerCallback):
    """
    Коллбэк для оценки качества генерации на тестовых промптах.
    Вызывается в конце каждой эпохи.
    """
    def __init__(self, tokenizer, model, prompts=questions_rus, max_length=50, device=None, sample_params=None):
        self.tokenizer = tokenizer
        self.model = model
        self.prompts = prompts
        self.max_length = max_length
        self.sample_params = sample_params
        
    def on_epoch_end(self, args, state, control, **kwargs):
        """
        Генерирует тексты для каждого промпта и выводит их.
        """
        device = next(self.model.parameters()).device
        temperature = self.sample_params.temperature if self.sample_params else 0.8
        top_p = self.sample_params.top_p if self.sample_params else 0.9
        do_sample = self.sample_params.do_sample if self.sample_params else True

        self.model.eval()
        with torch.no_grad():
            for prompt in self.prompts:
                input_ids = tokenizer\
                    .apply_chat_template(row2messages({"instruction": prompt}), return_tensors="pt")\
                    .to(device)
                output_ids = self.model.generate(
                    input_ids,
                    max_new_tokens=64,
                    temperature=temperature,
                    top_p=top_p,
                    do_sample=do_sample,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
                generated_ids = output_ids[0][input_ids.shape[1]:]
                generated_tokens = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
                print(f"Промпт:         {prompt}")
                print(f"Сгенерировано:  {''.join(generated_tokens)}\n")
        self.model.train()


class PerplexityCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            metrics["perplexity"] = math.exp(metrics["eval_loss"])
            print(f"Perplexity: {metrics['perplexity']:.2f}")


def run_sft(model, tokenizer, train_dataset, eval_dataset, sample_params):
    cfg = SFTConfig(
        output_dir="posttrained_sft",
        dataset_text_field = "text",
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        logging_steps=20,
        max_length=256,
        report_to='none',
        run_name='SFT',
        num_train_epochs=10,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-4,
        weight_decay=5e-2,
        load_best_model_at_end=True,
        optim="paged_adamw_8bit",
        fp16=True,
        packing=True,
        seed=42,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8,
    )
    prompt_callback = PromptEvalCallback(
        tokenizer,
        model,
        max_length=256,
        sample_params=sample_params,
    )
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.01
    )
    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        callbacks=[
            PerplexityCallback(),
            prompt_callback,
            early_stopping_callback,
        ],
    )
    trainer.train()

In [29]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("Выполнена очистка кеша CUDA")

run_sft(model, tokenizer, train_dataset, eval_dataset, SampleParams)

Выполнена очистка кеша CUDA
Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=7):   0%|          | 0/2700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=7):   0%|          | 0/300 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 10 | Total steps = 850
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)


Epoch,Training Loss,Validation Loss,
1,1.708900,1.733946,5.662954
2,1.629500,1.699146,5.469273
3,1.503000,1.695848,5.451269
4,1.380300,1.719470,5.581571


Промпт:         сколько планет в нашей солнечной системе?
Сгенерировано:  믑assistant
В нашей солнечной системе 8 планет. Это Земля, Сатурн, Юпитер, Венера, Марс, Три созвездия, Юпитер и Марс. С другой стороны, в 1994 году было

Промпт:         расскажи стих
Сгенерировано:  	NdrFcassistant
В глубине сердца я чувствовал себя как гибка,
Я чувствовал себя как столько-нибудь холостячика,
Вот, как я страстно жил, не оглядываясь,
И я жил, и я

Промпт:         когда собирать крыжовник?
Сгенерировано:  㑊assistant
Крыжовник – это пакет из 10-15 палочек, которые обычно бывают в разнообразии от дешевых до дорогих. Чтобы собирать крыжовник, вам необходимо собрать все необходимое набор материала и под

Промпт:         Как быстро выучить новый язык?
Сгенерировано:   """",
assistant
Для обучения нового языка можно использовать различные методы, такие как генерация, онлайн-курсы, учебники и практика. Одним из наиболее эффективных способов, который можно использовать для изучения нового языка, является 

In [32]:
test_prompts(model, tokenizer, questions_rus, 256, SampleParams)

---
Промпт:         сколько планет в нашей солнечной системе?
Чат:
user
сколько планет в нашей солнечной системе?
 לחלוטassistant
Солнечная система состоит из 8 планет, в том числе три созвездия: Земля, Венера и Марс. Их же называют солнечными планетами, поскольку они солнечные созвездия. Это означает, что они имеют общую часть телу, и, следовательно, имеют общую массу. Всеми созвездиями называют небо. В то же время, все планеты созвездия являются небольшими частями неба, такими, как земля, Венера, Марс, Юпитер и Сатурн. Все они являются созвездиями, и имеют общую телу, хотя они имеют разные расстояния от Солнца. В среднем планета отращивает около 29.53 облака вокруг Солнца. Один из трех созвездий, Солнечная Скаля, также называется Солнечной Скатерой. Все планеты, которые окружены Солнцем, имеют прямую сеч

---
Промпт:         расскажи стих
Чат:
user
расскажи стих
큠assistant
Тема дня: Делаем свою жизнь лучше

Предложите себе несколько способов, чтобы сделать свою жизнь лучше, с помощью

### Замечания
Заметно что модель уловила необхоидимость QA сценарий и генерация начинает напоминать ответы на вопросы. Если проучить её на бОльшем кол-ве примеров и сделать полный SFT это будет выглядеть как нормальный чатбот.

В рамках практики у меня оставался запуск сервера с A100. Там я попробовал обучить полный SFT и это выглядело намного лучше, только осталось в другом ноутбуке.